# 2. Streaming Conformer：上下文与缓存

**研究问题**：一帧到达时，哪些历史张量需要保留，分块输出何时能与同一因果模型的整段计算一致？本本 notebook 使用独立教学模块，不调用第一本 notebook。下采样可先由第一本完成；此处输入 `x:[B,T,D]` 表示已下采样的帧。**有限左上下文**为最近 $L$ 个历史帧，当前 chunk 大小为 $C$，无未来可见帧。

## 整体架构图




In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(11)
B, T, D, H, L, C, K = 2, 13, 16, 4, 5, 4, 3

## 1. 因果注意力与 KV 缓存

第 $t$ 帧可见的 key 索引满足 $\max(0,t-L)\le j\le t$，其中 $L$ 是**历史帧数**。每头 $q_t,k_j,v_j\in\mathbb R^{d_h}$，$\alpha_{t,j}=\mathrm{softmax}_{j}(q_tk_j^\top/\sqrt{d_h})$，$o_t=\sum_j\alpha_{t,j}v_j$。

对 chunk `[B,C,D]`，投影后当前 `q,k,v:[B,H,C,d_h]`。和历史 `K,V:[B,H,≤L,d_h]` 拼接后是 `[B,H,≤L+C,d_h]`，score 为 `[B,H,C,≤L+C]`。掩码同时排除未来帧和超出左上下文的帧。处理后只保留末尾 $L$ 帧的 K/V。位置由全局 `offset` 决定，避免每个 chunk 从零重新计数。

### 实际使用的带状下三角掩码

下面以 $T=6$、左上下文 $L=2$ 为例。**行**是当前查询帧 $t$，**列**是候选 key 帧 $j$；主对角线 `◆` 表示当前帧看自己，`●` 表示可见的最近历史，`·` 表示超出左上下文的历史，`×` 表示未来帧。

```text
                     key 帧 j →
查询帧 t ↓          0   1   2   3   4   5
              0     ◆   ×   ×   ×   ×   ×
              1     ●   ◆   ×   ×   ×   ×
              2     ●   ●   ◆   ×   ×   ×
              3     ·   ●   ●   ◆   ×   ×
              4     ·   ·   ●   ●   ◆   ×
              5     ·   ·   ·   ●   ●   ◆
```

允许关注的格子满足 $\max(0,t-L)\le j\le t$。例如第 4 帧只看第 2、3、4 帧。实现中先得到当前 chunk 与缓存拼接后的分数 `[B,H,C,≤L+C]`，对 `·` 和 `×` 位置在 softmax 前施加掩码，使其权重为 0；当前 chunk 内的未来帧同样被屏蔽。


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, dim, heads, left):
        super().__init__()
        assert dim % heads == 0 and left >= 0
        self.heads, self.left = heads, left
        self.qkv, self.out = nn.Linear(dim, 3*dim), nn.Linear(dim, dim)
    def step(self, x, cache=None, offset=0, trace=False):
        b, c, d = x.shape; dh = d // self.heads
        q,k,v = self.qkv(x).reshape(b,c,3,self.heads,dh).permute(2,0,3,1,4)
        old_k,old_v = cache if cache is not None else (k[:,:,:0],v[:,:,:0])
        keys,values = torch.cat([old_k,k],2),torch.cat([old_v,v],2)
        key_pos = torch.arange(offset-old_k.shape[2],offset+c,device=x.device)
        query_pos = torch.arange(offset,offset+c,device=x.device)
        visible = (key_pos[None,:] <= query_pos[:,None]) & (key_pos[None,:] >= query_pos[:,None]-self.left)
        scores = q @ keys.transpose(-2,-1) / math.sqrt(dh)
        scores = scores.masked_fill(~visible[None,None],torch.finfo(scores.dtype).min)
        weights = scores.softmax(-1)
        y = self.out((weights @ values).transpose(1,2).reshape(b,c,d))
        new_cache = (keys[:,:,-self.left:] if self.left else keys[:,:,:0], values[:,:,-self.left:] if self.left else values[:,:,:0])
        shapes = {'query':tuple(q.shape),'kv_concat':tuple(keys.shape),'scores':tuple(scores.shape),'cache':tuple(new_cache[0].shape)}
        return (y,new_cache,shapes) if trace else (y,new_cache)
    def forward(self,x): return self.step(x)[0]

att = CausalAttention(D,H,L).eval()
a,kv,shape_info = att.step(torch.randn(B,C,D),trace=True)
print(shape_info)
assert a.shape == (B,C,D) and kv[0].shape == (B,H,C,D//H)

## 2. 因果卷积与卷积缓存

这里使用一维深度卷积 $y_{t,c}=\sum_{r=0}^{K-1}w_{c,r}x_{t-(K-1)+r,c}$，只依赖当前及过去 $K-1$ 帧。GLU 前后的形状为 `[B,2D,C] → [B,D,C]`。旧缓存 `[B,D,K-1]` 与当前 GLU 输出 `[B,D,C]` 拼接后为 `[B,D,K-1+C]`，**不补右侧零**的卷积输出恰为 `[B,D,C]`。

与第一本的对称填充和 BatchNorm 不同，这里选因果卷积与逐帧 LayerNorm；这样整段和分块的运算语义一致。

In [ ]:
class CausalConv(nn.Module):
    def __init__(self, dim, kernel):
        super().__init__()
        self.kernel=kernel
        self.norm=nn.LayerNorm(dim)
        self.pw1=nn.Conv1d(dim,2*dim,1)
        self.dw=nn.Conv1d(dim,dim,kernel,groups=dim)
        self.pw2=nn.Conv1d(dim,dim,1)
    def step(self,x,cache=None,trace=False):
        z=F.glu(self.pw1(self.norm(x).transpose(1,2)),dim=1)
        if cache is None: cache=z.new_zeros(z.shape[0],z.shape[1],self.kernel-1)
        joined=torch.cat([cache,z],dim=-1)
        y=self.pw2(F.silu(self.dw(joined))).transpose(1,2)
        new_cache=joined[:,:,-(self.kernel-1):] if self.kernel>1 else joined[:,:,:0]
        shapes={'glu':tuple(z.shape),'joined':tuple(joined.shape),'output':tuple(y.shape),'cache':tuple(new_cache.shape)}
        return (y,new_cache,shapes) if trace else (y,new_cache)
    def forward(self,x): return self.step(x)[0]

conv=CausalConv(D,K).eval()
y,cc,conv_shapes=conv.step(torch.randn(B,C,D),trace=True)
print(conv_shapes)
assert y.shape==(B,C,D) and cc.shape==(B,D,K-1)

## 3. 组合流式 block，并与整段计算比较

仍采用 Macaron 顺序：$x_1=x+\tfrac12F_1(x)$，$x_2=x_1+A(x_1)$，$x_3=x_2+C(x_2)$，$y=\mathrm{LN}(x_3+\tfrac12F_2(x_3))$。FFN 和 LayerNorm 只沿通道作用。注意力缓存必须保存 **FFN1 之后、注意力投影所得的 K/V**；卷积缓存保存 **注意力残差之后、GLU 所得的输入**。两套缓存属于不同位置，不能互换。

整段基准使用同一个 `step` 一次处理所有帧，因此不是拿无限上下文模型作比较。时间轴例：`[0..3] → [4..7] → [8..11] → [12]`；最后一块不足 $C$ 时直接用实际长度。

In [ ]:
class StreamFFN(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.net=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,4*dim),nn.SiLU(),nn.Linear(4*dim,dim))
    def forward(self,x): return self.net(x)

class StreamingBlock(nn.Module):
    def __init__(self,dim,heads,left,kernel):
        super().__init__()
        self.ff1,self.ff2=StreamFFN(dim),StreamFFN(dim)
        self.attn=CausalAttention(dim,heads,left)
        self.conv=CausalConv(dim,kernel)
        self.norm=nn.LayerNorm(dim)
    def step(self,x,state=None,offset=0):
        old_kv,old_conv=state if state is not None else (None,None)
        z=x+0.5*self.ff1(x)
        a,new_kv=self.attn.step(z,old_kv,offset)
        z=z+a
        c,new_conv=self.conv.step(z,old_conv)
        z=z+c
        return self.norm(z+0.5*self.ff2(z)),(new_kv,new_conv)
    def forward(self,x): return self.step(x)[0]

block=StreamingBlock(D,H,L,K).eval()
x=torch.randn(B,T,D,requires_grad=True)
whole=block(x)
state=None; pieces=[]
for start in range(0,T,C):
    out,state=block.step(x[:,start:start+C],state,offset=start)
    pieces.append(out)
chunked=torch.cat(pieces,dim=1)
max_err=(whole-chunked).abs().max().item()
print('整段/分块形状:',whole.shape,chunked.shape,'最大误差:',max_err)
print('最终 KV/Conv 缓存:',state[0][0].shape,state[1].shape)
assert whole.shape==chunked.shape==(B,T,D)
assert torch.allclose(whole,chunked,atol=2e-6,rtol=2e-6)
assert state[0][0].shape==(B,H,L,D//H) and state[1].shape==(B,D,K-1)
chunked.square().mean().backward()
assert x.grad is not None and torch.isfinite(x.grad).all()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in block.parameters())
print('等价、缓存及梯度检查通过')

**适用边界**：误差检查只证明此简化因果 block 在同一权重、同一左上下文和 eval 模式下的分块一致性。它不测真实麦克风延迟，也不代表 icefall 的完整 streaming Zipformer 或原版 Conformer。推理时可将缓存张量 `detach`；这里保留计算图用于梯度检查。